# Controle do VINI via Manete USB no CoppeliaSim

Este notebook separa a **instalação**, a **conexão** e o **loop de controle em tempo real** em células distintas para facilitar testes de velocidade e ajustes sem precisar reconectar tudo.

### Célula 1: Instalar as bibliotecas (rode apenas uma vez)

In [1]:
%pip install pygame-ce coppeliasim-zmqremoteapi-client

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


### Célula 2: Conectar a manete e o CoppeliaSim
Antes de rodar esta célula, deixe o **CoppeliaSim aberto com a cena em Play** (e com o script Lua antigo desativado) e a **manete plugada no USB**.

In [2]:
import time
import pygame
from coppeliasim_zmqremoteapi_client import RemoteAPIClient

# Inicializa a manete
pygame.init()
pygame.joystick.init()

if pygame.joystick.get_count() == 0:
    raise RuntimeError("Nenhuma manete encontrada! Plugue o controle e rode esta célula novamente.")

joy = pygame.joystick.Joystick(0)
joy.init()
print(f"Controle conectado: {joy.get_name()}")

# Conecta ao CoppeliaSim
client = RemoteAPIClient()
sim = client.require('sim')

# Localiza as juntas na cena atual (/base_link_respondable ou /VINI)
try:
    motor = sim.getObject('/base_link_respondable/rear_wheel_joint')
    steer = sim.getObject('/base_link_respondable/steering_joint')
except Exception:
    motor = sim.getObject('/VINI/rear_wheel_joint')
    steer = sim.getObject('/VINI/steering_joint')

# Inicia a simulação automaticamente caso ainda não esteja em Play
if sim.getSimulationState() == sim.simulation_stopped:
    sim.startSimulation()
    print("Simulação iniciada automaticamente no CoppeliaSim!")

print(f"Conectado ao CoppeliaSim com sucesso! (motor={motor}, steer={steer})")

pygame-ce 2.5.8 (SDL 2.32.10, Python 3.14.6)
Controle conectado: Xbox 360 Controller


C:\Users\Gabriel\AppData\Local\Temp\ipykernel_23404\1406345286.py:13: DeprecationWarning: pygame.joystick.Joystick.init() is deprecated since 2.4.0. Bear in mind in future it will not be possible to reinitialise a closed Joystick object.
  joy.init()


Conectado ao CoppeliaSim com sucesso! (motor=1482, steer=1485)


### Célula 3: Rodar o controle em tempo real
Ao rodar esta célula, ela ficará executando (`[*]`) enquanto você dirige. Para parar, clique no botão **Stop / Interrupt the kernel (■)** na barra superior do Jupyter — o `except KeyboardInterrupt` vai frear o robô automaticamente.

In [ ]:
VEL_MAX = 50.0      # Velocidade máxima (rad/s)
ANGULO_MAX = 0.60   # Limite da direção (~34 graus; diminua se quiser curvas menos fechadas)
DEADZONE = 0.15     # Zona morta do analógico
SUAVIZACAO = 0.10   # Velocidade da direção (0.05 = bem lenta/suave | 0.25 = mais rápida)

def aplicar_deadzone(valor, limite=DEADZONE):
    return 0.0 if abs(valor) < limite else valor

angulo_atual = 0.0

print("Dirigindo! Analógico esquerdo = direção suave | RT = frente | LT = ré.")
print("Clique no botão Stop (■) do Jupyter para encerrar.")

try:
    while True:
        pygame.event.pump()

        # Analógico esquerdo (horizontal) apenas para direção
        eixo_direcao = aplicar_deadzone(joy.get_axis(0))
        angulo_alvo = -eixo_direcao * ANGULO_MAX

        # Aproxima o ângulo atual do alvo aos poucos (reduz a velocidade de giro)
        angulo_atual += (angulo_alvo - angulo_atual) * SUAVIZACAO

        # Gatilhos LT (eixo 4) e RT (eixo 5): converte de [-1, 1] para [0, 1]
        lt = (joy.get_axis(4) + 1.0) / 2.0
        rt = (joy.get_axis(5) + 1.0) / 2.0

        if lt < 0.05: lt = 0.0
        if rt < 0.05: rt = 0.0

        # RT vai para frente (-), LT vai para trás (+)
        vel = (lt - rt) * VEL_MAX

        sim.setJointTargetVelocity(motor, vel)
        sim.setJointTargetPosition(steer, angulo_atual)

        time.sleep(0.02)  # 50 Hz

except KeyboardInterrupt:
    sim.setJointTargetVelocity(motor, 0.0)
    sim.setJointTargetPosition(steer, 0.0)
    print("Controle encerrado e robô parado.")

Dirigindo! Analógico esquerdo = direção suave | RT = frente | LT = ré.
Clique no botão Stop (■) do Jupyter para encerrar.
